In [6]:
# quick sar modeling for herg
# regression + classification w rdkit fp + small desc
# proper holdout test + 5 fold cv on train only

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, average_precision_score, f1_score,
    confusion_matrix, precision_recall_fscore_support,
    brier_score_loss
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve

import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit.Chem import DataStructs

# config
data_path = "herg_postprocessed.csv" 

smiles_col = "canonical_smiles"
y_col = "pchembl_value_num"
rel_col = "standard_relation"
type_col = "standard_type"

allowed_types = {"IC50", "Ki"}   # keep potency endpoints
allowed_rel = {"="}             
active_thr = 5.0                # pIC50 >= 5 => active

fp_radius = 2
fp_nbits = 2048

random_seed = 42


# load data
def load_measured_subset(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    keep = (
        df[smiles_col].notna()
        & df[y_col].notna()
        & df[rel_col].astype(str).str.strip().isin(allowed_rel)
        & df[type_col].astype(str).str.strip().isin(allowed_types)
    )

    out = df.loc[keep, [smiles_col, y_col]].copy()
    out = out.rename(columns={smiles_col: "smiles", y_col: "y"})
    out["y"] = pd.to_numeric(out["y"], errors="coerce")
    out = out.dropna(subset=["y"]).reset_index(drop=True)
    out = out[out["smiles"].astype(str).str.len() > 0].reset_index(drop=True)
    return out


# features
morgan_gen = GetMorganGenerator(radius=fp_radius, fpSize=fp_nbits)

desc_names = ["mw", "logp", "tpsa", "hbd", "hba", "rot_bonds", "ring_count"]

def featurize_one(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None

    fp = morgan_gen.GetFingerprint(mol)
    fp_arr = np.zeros((fp_nbits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, fp_arr)

    desc = np.array([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.RingCount(mol),
    ], dtype=np.float32)

    return fp_arr.astype(np.float32), desc


def build_features(df: pd.DataFrame):
    fps, descs, ys = [], [], []
    bad = 0
    for s, y in zip(df["smiles"].values, df["y"].values):
        fp, desc = featurize_one(s)
        if fp is None:
            bad += 1
            continue
        fps.append(fp)
        descs.append(desc)
        ys.append(float(y))

    if bad > 0:
        print("dropped", bad, "rows due to rdkit parse fail")

    return (
        np.asarray(fps, dtype=np.float32),
        np.asarray(descs, dtype=np.float32),
        np.asarray(ys, dtype=np.float32),
    )


def stack_features(fps, descs, mode: str):
    if mode == "fp":
        return fps
    if mode == "desc":
        return descs
    if mode == "fp_desc":
        return np.concatenate([fps, descs], axis=1)
    raise ValueError("unknown feature mode: " + mode)


def make_feature_names(mode: str):
    if mode == "fp":
        return [f"fp_bit_{i}" for i in range(fp_nbits)]
    if mode == "desc":
        return list(desc_names)
    if mode == "fp_desc":
        return [f"fp_bit_{i}" for i in range(fp_nbits)] + list(desc_names)
    raise ValueError("unknown feature mode")


# metrics
def reg_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "rmse": float(np.sqrt(mse)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


def cls_metrics(y_true, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(
        y_true, pred, average="binary", zero_division=0
    )
    return {
        "roc_auc": float(roc_auc_score(y_true, proba)),
        "pr_auc": float(average_precision_score(y_true, proba)),
        "precision": float(pr),
        "recall": float(rc),
        "f1": float(f1),
        "brier": float(brier_score_loss(y_true, proba)),
        "cm": confusion_matrix(y_true, pred),
    }


# cross validation
def cv_regression(model, x, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_seed)
    rows = []
    for fold, (tr_idx, va_idx) in enumerate(kf.split(x), start=1):
        x_tr, x_va = x[tr_idx], x[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        model.fit(x_tr, y_tr)
        pred = model.predict(x_va)
        m = reg_metrics(y_va, pred)
        m["fold"] = fold
        rows.append(m)
    return pd.DataFrame(rows)


def cv_classification(model, x, y, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_seed)
    rows = []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(x, y), start=1):
        x_tr, x_va = x[tr_idx], x[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        model.fit(x_tr, y_tr)
        proba = model.predict_proba(x_va)[:, 1]
        m = cls_metrics(y_va, proba)
        m["fold"] = fold
        rows.append({k: v for k, v in m.items() if k != "cm"})
    return pd.DataFrame(rows)


# plots
def plot_roc(y_true, proba, title, out_png):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_true, proba)
    plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1])
    plt.xlabel("false positive rate")
    plt.ylabel("true positive rate")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()


def plot_calibration(y_true, proba, title, out_png, n_bins=10):
    frac_pos, mean_pred = calibration_curve(
        y_true, proba, n_bins=n_bins, strategy="uniform"
    )
    plt.figure()
    plt.plot(mean_pred, frac_pos, marker="o")
    plt.plot([0, 1], [0, 1])
    plt.xlabel("mean predicted prob")
    plt.ylabel("fraction positives")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()


# importance
def top_perm_importance(estimator, x_test, y_test, feature_names, task: str, n_top=20):
    if task == "reg":
        scoring = "neg_root_mean_squared_error"
    else:
        scoring = "roc_auc"

    imp = permutation_importance(
        estimator,
        x_test,
        y_test,
        n_repeats=10,
        random_state=random_seed,
        scoring=scoring,
        n_jobs=-1
    )

    importances = imp.importances_mean
    order = np.argsort(-importances)[:n_top]

    rows = []
    for idx in order:
        rows.append({"feature": feature_names[idx], "importance": float(importances[idx])})
    return pd.DataFrame(rows)


# main
def main():
    df = load_measured_subset(data_path)
    print("rows after modeling filter:", len(df))
    print(df["y"].describe())

    fps, descs, y = build_features(df)
    y_bin = (y >= active_thr).astype(int)
    print("\nclass balance (fraction active):", float(y_bin.mean()))

    idx_all = np.arange(len(y))

    # proper holdout test
    idx_tr_reg, idx_te_reg = train_test_split(
        idx_all, test_size=0.2, random_state=random_seed
    )
    idx_tr_cls, idx_te_cls = train_test_split(
        idx_all, test_size=0.2, random_state=random_seed, stratify=y_bin
    )

    feature_modes = ["fp", "desc", "fp_desc"]

    # regression
    print("\n--- regression: predict pchembl_value_num ---")
    reg_rows = []

    # baseline mean + cv
    base_reg = DummyRegressor(strategy="mean")
    for mode in feature_modes:
        x = stack_features(fps, descs, mode)
        x_tr, x_te = x[idx_tr_reg], x[idx_te_reg]
        y_tr, y_te = y[idx_tr_reg], y[idx_te_reg]

        cv_df = cv_regression(base_reg, x_tr, y_tr, n_splits=5)
        cv_summary = cv_df[["rmse", "mae", "r2"]].agg(["mean", "std"])

        base_reg.fit(x_tr, y_tr)
        pred = base_reg.predict(x_te)
        m = reg_metrics(y_te, pred)

        reg_rows.append({
            "model": "baseline_mean",
            "features": mode,
            **m,
            "cv_rmse_mean": float(cv_summary.loc["mean", "rmse"]),
            "cv_rmse_std": float(cv_summary.loc["std", "rmse"]),
            "cv_mae_mean": float(cv_summary.loc["mean", "mae"]),
            "cv_mae_std": float(cv_summary.loc["std", "mae"]),
            "cv_r2_mean": float(cv_summary.loc["mean", "r2"]),
            "cv_r2_std": float(cv_summary.loc["std", "r2"]),
        })

    # elasticnet regression + cv
    reg_model = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("enet", ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=random_seed, max_iter=10000))
    ])

    for mode in feature_modes:
        x = stack_features(fps, descs, mode)
        x_tr, x_te = x[idx_tr_reg], x[idx_te_reg]
        y_tr, y_te = y[idx_tr_reg], y[idx_te_reg]

        cv_df = cv_regression(reg_model, x_tr, y_tr, n_splits=5)
        cv_summary = cv_df[["rmse", "mae", "r2"]].agg(["mean", "std"])

        reg_model.fit(x_tr, y_tr)
        pred = reg_model.predict(x_te)
        m = reg_metrics(y_te, pred)

        reg_rows.append({
            "model": "elasticnet",
            "features": mode,
            **m,
            "cv_rmse_mean": float(cv_summary.loc["mean", "rmse"]),
            "cv_rmse_std": float(cv_summary.loc["std", "rmse"]),
            "cv_mae_mean": float(cv_summary.loc["mean", "mae"]),
            "cv_mae_std": float(cv_summary.loc["std", "mae"]),
            "cv_r2_mean": float(cv_summary.loc["mean", "r2"]),
            "cv_r2_std": float(cv_summary.loc["std", "r2"]),
        })

    reg_table = pd.DataFrame(reg_rows)
    print("\nreg performance table (test + cv)")
    print(reg_table)

    # importance for regression (rf + permutation) on fp_desc
    print("\nreg feature importance (permutation) using randomforest on fp_desc")
    x_reg = stack_features(fps, descs, "fp_desc")
    x_tr, x_te = x_reg[idx_tr_reg], x_reg[idx_te_reg]
    y_tr, y_te = y[idx_tr_reg], y[idx_te_reg]

    rf_reg = RandomForestRegressor(
        n_estimators=400,
        random_state=random_seed,
        n_jobs=-1
    )
    rf_reg.fit(x_tr, y_tr)

    reg_imp = top_perm_importance(
        rf_reg, x_te, y_te,
        feature_names=make_feature_names("fp_desc"),
        task="reg",
        n_top=20
    )
    print(reg_imp)


    # classification
    print("\n--- classification: active if p >= 5 ---")
    cls_rows = []

    # baseline majority + cv
    base_cls = DummyClassifier(strategy="most_frequent")
    for mode in feature_modes:
        x = stack_features(fps, descs, mode)
        x_tr, x_te = x[idx_tr_cls], x[idx_te_cls]
        y_tr, y_te = y_bin[idx_tr_cls], y_bin[idx_te_cls]

        cv_df = cv_classification(base_cls, x_tr, y_tr, n_splits=5)
        cv_summary = cv_df[["roc_auc", "pr_auc", "f1", "brier"]].agg(["mean", "std"])

        base_cls.fit(x_tr, y_tr)
        proba = base_cls.predict_proba(x_te)[:, 1]
        m = cls_metrics(y_te, proba)

        cls_rows.append({
            "model": "baseline_majority",
            "features": mode,
            **{k: v for k, v in m.items() if k != "cm"},
            "cv_roc_auc_mean": float(cv_summary.loc["mean", "roc_auc"]),
            "cv_roc_auc_std": float(cv_summary.loc["std", "roc_auc"]),
            "cv_pr_auc_mean": float(cv_summary.loc["mean", "pr_auc"]),
            "cv_pr_auc_std": float(cv_summary.loc["std", "pr_auc"]),
            "cv_f1_mean": float(cv_summary.loc["mean", "f1"]),
            "cv_f1_std": float(cv_summary.loc["std", "f1"]),
            "cv_brier_mean": float(cv_summary.loc["mean", "brier"]),
            "cv_brier_std": float(cv_summary.loc["std", "brier"]),
        })

    # logreg classifier + cv
    cls_model = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("lr", LogisticRegression(
            max_iter=4000,
            class_weight="balanced",
            random_state=random_seed
        ))
    ])

    for mode in feature_modes:
        x = stack_features(fps, descs, mode)
        x_tr, x_te = x[idx_tr_cls], x[idx_te_cls]
        y_tr, y_te = y_bin[idx_tr_cls], y_bin[idx_te_cls]

        cv_df = cv_classification(cls_model, x_tr, y_tr, n_splits=5)
        cv_summary = cv_df[["roc_auc", "pr_auc", "f1", "brier"]].agg(["mean", "std"])

        cls_model.fit(x_tr, y_tr)
        proba = cls_model.predict_proba(x_te)[:, 1]
        m = cls_metrics(y_te, proba)

        cls_rows.append({
            "model": "logreg_balanced",
            "features": mode,
            **{k: v for k, v in m.items() if k != "cm"},
            "cv_roc_auc_mean": float(cv_summary.loc["mean", "roc_auc"]),
            "cv_roc_auc_std": float(cv_summary.loc["std", "roc_auc"]),
            "cv_pr_auc_mean": float(cv_summary.loc["mean", "pr_auc"]),
            "cv_pr_auc_std": float(cv_summary.loc["std", "pr_auc"]),
            "cv_f1_mean": float(cv_summary.loc["mean", "f1"]),
            "cv_f1_std": float(cv_summary.loc["std", "f1"]),
            "cv_brier_mean": float(cv_summary.loc["mean", "brier"]),
            "cv_brier_std": float(cv_summary.loc["std", "brier"]),
        })

        print("\nlogreg test cm (" + mode + ")")
        print(m["cm"])

        plot_roc(y_te, proba, f"roc curve logreg ({mode})", f"roc_{mode}.png")
        plot_calibration(y_te, proba, f"calibration logreg ({mode})", f"cal_{mode}.png")

    cls_table = pd.DataFrame(cls_rows)
    print("\ncls performance table (test + cv)")
    print(cls_table)

    # importance for classification (permutation) on fp_desc
    print("\ncls feature importance (permutation) for logreg on fp_desc")
    x_cls = stack_features(fps, descs, "fp_desc")
    x_tr, x_te = x_cls[idx_tr_cls], x_cls[idx_te_cls]
    y_tr, y_te = y_bin[idx_tr_cls], y_bin[idx_te_cls]

    cls_model.fit(x_tr, y_tr)
    cls_imp = top_perm_importance(
        cls_model, x_te, y_te,
        feature_names=make_feature_names("fp_desc"),
        task="cls",
        n_top=20
    )
    print(cls_imp)

    # save outputs for report
    reg_table.to_csv("results_regression_table.csv", index=False)
    cls_table.to_csv("results_classification_table.csv", index=False)
    reg_imp.to_csv("results_regression_importance_top20.csv", index=False)
    cls_imp.to_csv("results_classification_importance_top20.csv", index=False)

    print("\nsaved outputs")
    print("results_regression_table.csv")
    print("results_classification_table.csv")
    print("results_regression_importance_top20.csv")
    print("results_classification_importance_top20.csv")
    print("roc_*.png and cal_*.png")


if __name__ == "__main__":
    main()

rows after modeling filter: 1221
count    1221.00000
mean        5.84973
std         1.14472
min         4.00000
25%         5.00000
50%         5.65000
75%         6.58000
max         9.82000
Name: y, dtype: float64

class balance (fraction active): 0.7583947583947583

--- regression: predict pchembl_value_num ---

reg performance table (test + cv)
           model features      rmse       mae        r2  cv_rmse_mean  \
0  baseline_mean       fp  1.122752  0.921296 -0.001310      1.149085   
1  baseline_mean     desc  1.122752  0.921296 -0.001310      1.149085   
2  baseline_mean  fp_desc  1.122752  0.921296 -0.001310      1.149085   
3     elasticnet       fp  0.852548  0.572904  0.422651      0.851999   
4     elasticnet     desc  1.190686  0.904033 -0.126147      1.473480   
5     elasticnet  fp_desc  0.840244  0.558135  0.439195      1.001334   

   cv_rmse_std  cv_mae_mean  cv_mae_std  cv_r2_mean  cv_r2_std  
0     0.056883     0.936499    0.053324   -0.002991   0.004080  
1     